In [ ]:
### Imports

import pandas as pd
import geopandas as gpd
import hvplot.pandas as hv
import geoviews as gv
import json

SalesShape = gpd.read_file("../src/data/Geospacial/SalesTaxAreas.shp")
SalesRateComp = pd.read_excel("../src/data/Tax&Spend/SalesRateComponents.xlsx", sheet_name="MachineReadable")

PropertyShape = gpd.read_file("../src/data/Geospacial/PropertyTaxEntities2025.shp")

In [ ]:
### Notes

## Property shape
    # Figure out mapping from Shape file entities to certifed rate system 

In [ ]:
### Preproccessing

counties_dict = {'01' : "Beaver",
    '02' : "Box Elder",
    '03' : "Cache",
    '04' : "Carbon",
    '05' : "Dagget",
    '06' : "Davis",
    '07' : "Duchesne",
    '08' : "Emery",
    '09' : "Garfield",
    '10' : "Grand",
    '11' : "Iron",
    '12' : "Juab",
    '13' : "Kane",
    '14' : "Millard",
    '15' : "Morgan",
    '16' : "Piute",
    '17' : "Rich",
    '18' : "Salt Lake",
    '19' : "San Juan",
    '20' : "Sanpete",
    '21' : "Sevier",
    '22' : "Summit",
    '23' : "Tooele",
    '24' : "Uintah",
    '25' : "Utah",
    '26' : "Wasatch",
    '27' : "Washington",
    '28' : "Wayne",
    '29' : "Weber"
} 

In [ ]:
### Property Shape Data Cleaning

# Drop needless columns
PropertyShape = PropertyShape.drop(columns = ['ENT_ID','ENT_CODE', 'created_us', 'created_da', 'last_edite', 'last_edi_1'])


# Create explicit county variable
PropertyShape['county'] = PropertyShape["ENT_CO"].map(counties_dict)

# Create explicit entity type variable
PropertyShape.loc[PropertyShape['ENT_NBR'] == 1010, 'entity_type'] = 'County'
PropertyShape.loc[PropertyShape['ENT_NBR'] == 1015, 'entity_type'] = 'Multicounty Assessing'
PropertyShape.loc[PropertyShape['ENT_NBR'] == 1020, 'entity_type'] = 'County Assessing'
PropertyShape.loc[(PropertyShape['ENT_NBR'] >= 2000) & (PropertyShape['ENT_NBR'] < 3000), 'entity_type'] = 'School District'
PropertyShape.loc[(PropertyShape['ENT_NBR'] >= 3000) & (PropertyShape['ENT_NBR'] < 5000), 'entity_type'] = 'Municipality'
PropertyShape.loc[(PropertyShape['ENT_NBR'] >= 5000) & (PropertyShape['ENT_NBR'] < 6000), 'entity_type'] = 'PID'

PropertyShape.loc[(PropertyShape['ENT_NBR'] >= 6000) & (PropertyShape['ENT_NBR'] < 7000), 'entity_type'] = 'Special District'

PropertyShape.loc[(PropertyShape['ENT_NBR'] >= 8000) & (PropertyShape['ENT_NBR'] < 9999), 'entity_type'] = 'RDA or CDA'
PropertyShape.loc[PropertyShape['ENT_NBR'] >= 9999, 'entity_type'] = 'Statewide'

# Set epsg for geoviews use
PropertyShape = PropertyShape.set_crs(epsg=3857)


In [ ]:
### Create negative buffer to counteract spurious overlaps
PropertyShapeBackup = PropertyShape.copy()

PropertyShape['geometry'] = PropertyShape['geometry'].buffer(-50)

empty_mask = PropertyShape['geometry'].is_empty

PropertyShape.loc[empty_mask, 'geometry'] = PropertyShapeBackup[empty_mask]


In [ ]:
### Sales Shape Data Cleaning

# Set epsg for geoviews use
SalesShape = SalesShape.set_crs(epsg=3857)

# Merge in Rate composition data
SalesRateComp['TAXDIST'] = SalesRateComp['TAXDIST'].astype(str).str.zfill(5)

diffValues = set(SalesRateComp['TAXDIST']) - set(SalesShape['TAXDIST'])

SalesRateCompSubset = SalesRateComp[~SalesRateComp['TAXDIST'].isin(diffValues)]

SalesShape = pd.merge(SalesShape, SalesRateComp, on='TAXDIST')



In [ ]:
### Create negative buffer to counteract spurious overlaps
SalesShapeBackup = SalesShape.copy()

SalesShape['geometry'] = SalesShape['geometry'].buffer(-50)

empty_mask = SalesShape['geometry'].is_empty

SalesShape.loc[empty_mask, 'geometry'] = SalesShapeBackup[empty_mask]


In [ ]:
### Testing

PropertyShapeFiltered = PropertyShape[(PropertyShape['ENT_NBR']== 1010)]

PropertyShapeFiltered.explore(color = 'green', tooltip='ENT_DESC')

#SalesShapeFiltered = SalesShape[~SalesShape["SHORTDESC_y"].str.contains('County')]

#SalesShapeFiltered = SalesShapeFiltered['geometry'].buffer(-50)

#SalesShapeFiltered.explore(color = 'red')

In [9]:
### Output to usable JSON file

PropertyShape.to_crs(epsg=4326).to_file('../src/data/Geospacial/Property2025.json', driver='GeoJSON')
SalesShape.to_crs(epsg=4326).to_file('../src/data/Geospacial/Sales2025.json', driver='GeoJSON')

In [10]:
### Generate District Intersection Lookup Table

# Read back the output files — indices here are guaranteed to match the JSON feature array
PropertyOut = gpd.read_file('../src/data/Geospacial/Property2025.json')
SalesOut    = gpd.read_file('../src/data/Geospacial/Sales2025.json')
HouseDistricts  = gpd.read_file('../src/data/Geospacial/HouseDistricts.json')
SenateDistricts = gpd.read_file('../src/data/Geospacial/SenateDistricts.json')

def build_lookup(features_gdf, districts_gdf, dist_col='DIST'):
    joined = gpd.sjoin(
        features_gdf[['geometry']].reset_index(),
        districts_gdf[[dist_col, 'geometry']],
        how='inner',
        predicate='intersects'
    )
    lookup = {}
    for dist, group in joined.groupby(dist_col):
        lookup[str(int(dist))] = sorted(group['index'].tolist())
    return lookup

table = {
    "house": {
        "property": build_lookup(PropertyOut, HouseDistricts),
        "sales":    build_lookup(SalesOut,    HouseDistricts),
    },
    "senate": {
        "property": build_lookup(PropertyOut, SenateDistricts),
        "sales":    build_lookup(SalesOut,    SenateDistricts),
    }
}

with open('../src/data/Geospacial/DistrictIntersections.json', 'w') as f:
    json.dump(table, f)

print(f"Done. {len(PropertyOut)} property features, {len(SalesOut)} sales features.")

Done. 1541 property features, 373 sales features.
